### 03_image_preprocessing

The central question for this notebook is:

**How do we convert a folder of raw product images into clean, consistent tensors that can be safely used for model training?**

The pipeline will eventually look like:

``` text

Raw images
    ↓
Validate images
    ↓
Train / Validation / Test split
    ↓
Resize
    ↓
Convert to tensor
    ↓
Normalize
    ↓
Training augmentation
    ↓
DataLoader
    ↓
CNN

```
There are several new concepts here, so we'll build them one at a time.

##### 1. Why do images need preprocessing?

Imagine our dataset contains:

- Image A → 640 × 480
- Image B → 1024 × 768
- Image C → 300 × 300
- Image D → 1920 × 1080

A neural network trains efficiently using batches, and the images within a batch need compatible dimensions.

We therefore need a consistent representation such as:

``` text

Image A ─┐
Image B ─┤
Image C ─┼──→ [3, 224, 224]
Image D ─┘

```
But preprocessing involves more than resizing.

We also need to think about:

- Image dimensions
- Pixel scale
- Color channels
- Image quality
- Class labels
- Dataset splits
- Augmentation
- Normalization

##### 2. First: inspect the dataset

Before transforming anything, we should understand what we actually have.

For a defect-classification dataset, we might have categories such as:

``` text

dataset/

scratch/
    image_001.jpg
    image_002.jpg

crack/
    image_003.jpg
    image_004.jpg

dent/
    image_005.jpg
    image_006.jpg
```
The folder name can represent the label:

``` text

scratch/image_001.jpg
       ↓
label = scratch

```

Before training, we should inspect:

- Number of images
- Number of classes
- Images per class
- Image dimensions
- Image formats
- Corrupted images
- Duplicate images

This isn't merely housekeeping.

**⚖️ Responsible AI (RAI) — Data quality**

The model learns from the data we provide.

If the dataset contains:

- Scratch → 2,000 images
- Crack   →   150 images
- Dent    →   100 images

then we immediately have a potential reliability issue.

A high overall accuracy could hide poor performance on minority defect classes.

So Responsible AI begins before model training.

##### ⚖️ RAI — Data provenance and label definition

Before preprocessing, we should also understand:

- where the dataset came from;
- how the images were collected;
- who or what created the labels;
- what each class label actually means;
- whether the labels were independently verified;
- whether the dataset is representative of the intended production use.

A technically valid image is not necessarily a trustworthy training sample.
Incorrect labels, unclear provenance, or unrepresentative collection
conditions can still produce an unreliable model.

##### 3. Train, validation and test split


``` text

Full dataset
      ↓
 ┌────┼─────┐
 ↓    ↓     ↓
Train Val   Test

```
For example:

- 70% Training
- 15% Validation
- 15% Test

The roles remain exactly the same as in your previous ML/NLP projects.

Training set teaches the model.

Validation set helps us evaluate the model while developing it.

Test set is intended to provide an independent final evaluation after development decisions are complete.

Why? Because we learned firsthand that simply calling something a test set does not guarantee unbiased or independent evaluation.

That's actually a beautiful connection between Notebook 03 and what happened later:

``` text

Notebook 03
───────────
Theory:
"Could related images leak across splits?"

             ↓

Notebook 11
───────────
Evidence:
64 exact development/test duplicates found

             ↓

Notebook 15
───────────
RAI assessment:
Quantify impact
99.58% → 99.54%

             ↓

Professional conclusion:
Document the limitation;
don't hide it and don't exaggerate it.

```

##### 4. ⚖️ RAI — Data leakage becomes especially interesting with images

Suppose we have:

- product_001_front.jpg
- product_001_side.jpg
- product_001_closeup.jpg

If we randomly split individual images, this could happen:

- Training: product_001_front.jpg
- Validation: product_001_side.jpg
- Test: product_001_closeup.jpg

Now the model may have effectively seen the same physical product during training.

**That's potential data leakage.**

The test score could therefore look better than real-world performance.

A safer split might be:

- Product 001 → TRAIN only
- Product 002 → TRAIN only
- Product 003 → VALIDATION only
- Product 004 → TEST only

This gives us an important RAI lesson:

**⚖️ RAI — Reliable evaluation requires splitting data according to the real-world unit of independence, not blindly splitting rows/files.**

For our actual dataset, we'll determine what splitting strategy is appropriate based on its structure.

**⚖️ RAI — Test-set independence must be verified**

A test folder should not automatically be assumed to be independent simply
because it was supplied separately from the training folder.

Before model development, we should check for overlap between the development
data and test data.

Possible overlap includes:

- exact byte-identical images;
- resized or recompressed copies;
- near-duplicate images;
- multiple views of the same physical product;
- sequential images from the same manufacturing event.

An exact file hash such as SHA-256 can identify byte-identical files, but it
cannot detect all forms of related or near-duplicate data.

Therefore:

    Separate folder
        ≠
    guaranteed independent test data

A reliable evaluation requires the test set to represent genuinely unseen
samples according to the real-world unit of independence.

##### 5. Resizing

Suppose we decide to use: 224 × 224

We can transform different images:

``` text

640 × 480
    ↓
224 × 224

1024 × 768
    ↓
224 × 224

```

In PyTorch/torchvision, we'll commonly use a transform such as: transforms.Resize((224, 224))

Eventually:

```

RGB image
224 × 224
     ↓
Tensor
[3, 224, 224]

```

##### 6. But resizing can distort images

There's an important subtlety.

Suppose the original image is: 800 × 400

That's a wide rectangle.

If we force it directly into: 224 × 224

we change its aspect ratio.

Something that originally looked: ────────────────

could become visually compressed or stretched.

This can matter because shape itself may contain useful information.

Alternatives include:

``` text

Resize while preserving aspect ratio
        +
Crop

or

Resize
        +
Padding

```

We don't automatically choose the most complicated option. We choose preprocessing appropriate for the dataset and model.

##### 7. Convert image to tensor

Once the image has the correct size, it must become a tensor.

Conceptually:

``` text

PIL image
224 × 224 RGB
      ↓
ToTensor()
      ↓
PyTorch tensor
[3, 224, 224]

```
Traditionally, you might see: transforms.ToTensor()


This does two important things for a standard 8-bit RGB image:

``` text

H × W × C
    ↓
C × H × W

```

and converts/scales pixel values approximately from: 0 ... 255 to: 0.0 ... 1.0

So:

``` text

PIL / NumPy

[224,224,3]
pixel range 0–255

        ↓

PyTorch tensor

[3,224,224]
pixel range 0.0–1.0

```

##### 8. Scaling vs normalization

This distinction is important.

**Scaling**

Dividing by 255 gives:

- 0 → 0.0
- 128 → approximately 0.502
- 255 → 1.0

That's scaling.

**Normalization**

Normalization often goes further.

A common formula is:

normalized value = (value - mean) / standard deviation

This should look familiar from StandardScaler.

For images, it is often done separately for each channel:

- Red: (R - mean_R) / std_R
- Green: (G - mean_G) / std_G
- Blue: (B - mean_B) / std_B

So:
``` text

RGB tensor
     ↓
Scale
     ↓
Channel normalization
     ↓
Model-ready tensor

```

##### 9. Why do pretrained models care about normalization?

This becomes especially important during transfer learning.

Suppose a pretrained model originally learned using images normalized in a particular way.

Then when we use that pretrained model, we generally want our inputs to follow the preprocessing expected by those pretrained weights.

Otherwise:

**Model learned from one input distribution but receives a different input distribution and performance may suffer.**

We'll revisit this carefully when we reach:

08_transfer_learning_foundations

and

09_pretrained_vision_model.

##### 10. Image augmentation

Now we reach one of the most useful CV concepts.

Suppose our training dataset contains this image: Original product image

Instead of always showing the exact same image, we can randomly create modified training versions:

``` text

Original
   ↓
Horizontal flip

Original
   ↓
Small rotation

Original
   ↓
Slight crop

Original
   ↓
Brightness adjustment

```
This is called: **Data augmentation**

The goal is not simply to create more files.

The goal is to teach the model: The class should remain correct despite reasonable variations in the image.

##### 11. Example: rotation

Suppose:

``` text
Original image
      ↓
Scratch

```
We rotate the image slightly: +8 degrees

It is still: Scratch

The model should not learn:

"A scratch only counts as a scratch when it is perfectly horizontal."

Instead, we want it to learn more general visual patterns.

##### 12. Augmentation can improve generalization

Without augmentation, a model might memorize characteristics of the training images:

- Specific orientation
- Specific lighting
- Specific position
- Specific framing

With appropriate augmentation:

- different orientations
- different positions
- reasonable lighting variations

the model may learn more robust visual features.

This helps **generalization**: performing well on new, unseen images.

##### 13. ⚖️ RAI — augmentation must be realistic

This is another Responsible AI point.

Not every augmentation makes sense for every business problem.

Imagine we perform:

- 180-degree rotation
- Extreme color change
- Heavy blur
- Extreme cropping

If those conditions would **never occur in the actual manufacturing process**, they may not be useful.

Even worse, an augmentation could change the meaning of the image.

The question should therefore be:

**Does this transformation represent a plausible real-world variation while preserving the correct label?**

That's a ⚖️ Responsible AI — reliability/data validity concern.

##### 14. Training transforms vs validation/test transforms

This is very important.

Suppose training uses:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

Every time an image is loaded during training, it may appear slightly differently.

That's intentional.

But validation and test should normally be deterministic.

For example:

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

Why don't we randomly rotate the test images?

Because we want stable evaluation:

``` text

Same test image
      ↓
Same preprocessing
      ↓
Comparable evaluation

```
So remember:

TRAIN
- Resize
- Random augmentation
- Tensor
- Normalize

VALIDATION
- Resize
- Tensor
- Normalize

TEST
- Resize
- Tensor
- Normalize

The random augmentation belongs primarily to training.

##### 15. Does augmentation permanently create new image files?

Usually, no.

This is an important concept.

Suppose our dataset has: 1,000 training images

We don't necessarily generate: 10,000 JPG files and save them.

Instead, augmentation is often performed on the fly when an image is loaded.

For example:

``` text

Epoch 1:
image_001
   ↓
rotation +3°

Epoch 2:
image_001
   ↓
horizontal flip

Epoch 3:
image_001
   ↓
rotation -5°

Epoch 4:
image_001
   ↓
almost unchanged

```

So the same original image can appear differently across epochs.

That's powerful.

##### 16. Dataset and DataLoader

You've already used DataLoader with PyTorch.

Computer Vision uses the same idea.

Conceptually:

``` text

Image files
     ↓
Dataset
     ↓
Transform image
     ↓
Return:
(image_tensor, label)
     ↓
DataLoader
     ↓
Create batches

```
For one image:

- image_tensor: [3,224,224]
- label: 2

For a batch of 32:

- images: [32,3,224,224]
- labels:[32]

Compare that with your NLP neural network:

NLP

- X: [16,384]
- y: [16]

Now:

Computer Vision

- X: [32,3,224,224]
- y:[32]

Same training concept. Different input representation.

##### 17. What happens during training?

Our future loop will eventually look conceptually like:

``` text
for images, labels in train_loader:

    outputs = model(images)

    loss = criterion(outputs, labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

```
This should look very familiar.

The major new piece is: 

images.shape : [batch, channels, height, width]

The rest of the neural-network training logic hasn't fundamentally changed.

##### 18. 🔍 XAI connection — preprocessing can affect explanations

Here's an important XAI point that we'll explore later.

Suppose Grad-CAM shows that the model focuses on: background

instead of: actual defect

That might indicate the model learned a shortcut.

For example, perhaps all scratch images happened to have a darker background.

The model may have learned: dark background → scratch

instead of:

``` text

scratch pattern → scratch

🔍 XAI can help reveal that.

And once we discover it...

🔍 XAI
"What did the model focus on?"
        ↓
Background
        ↓
⚖️ RAI
"Is this model reliable?"

```

This is a great example of XAI and RAI working together.

##### 19. ⚖️ RAI — Pre-Training Data & Preprocessing Checklist



Before training, ask:

##### Data provenance and labels

- Where did the images come from?
- How were the labels created?
- Is the meaning of each class clearly defined?
- Are labels trustworthy and sufficiently verified?

##### Data quality

- Are classes reasonably represented?
- Are there corrupted or unreadable images?
- Are image formats and color channels consistent?
- Are image resolutions very different?

##### Evaluation integrity

- Are there exact duplicate images?
- Are there near-duplicate or highly related images?
- Could images of the same physical product appear across splits?
- Is there overlap between development data and the test set?
- Does the split reflect the real-world unit of independence?

##### Production representativeness

- Do lighting and camera conditions vary?
- Do the available images resemble expected production images?
- Are important production conditions missing from the dataset?

##### Preprocessing

- Are augmentations realistic?
- Do augmentations preserve the correct label?
- Could resizing or cropping remove important defect information?
- Does normalization match the model's requirements?
- Will production inference use the same deterministic preprocessing?

These checks are not separate from ML engineering.

They are part of building a reliable AI system.

##### 20. Our eventual production preprocessing module

Because we're keeping our production-style structure, we won't leave reusable preprocessing scattered throughout notebooks.

We'll eventually have:

src/
└── image_preprocessing.py

with reusable logic for things such as:

- training transforms
- evaluation transforms
- inference transforms

Conceptually:

``` text

def build_train_transform():
    ...

def build_eval_transform():
    ...

def build_inference_transform():
    ...

```
This becomes especially important during deployment.

The endpoint must preprocess an incoming image consistently with training/inference requirements.

##### 21. ⚖️ RAI — training/serving consistency

Suppose training uses:

- Resize → 224×224
- Normalize → expected mean/std

but deployment accidentally uses:

- Resize → 256×256
- No normalization

The endpoint may produce unreliable predictions even though the model itself is correct.

Therefore:

**⚖️ RAI — Reliability includes maintaining consistent preprocessing between model development and production inference.**

This is one reason we're putting preprocessing in reusable src/ code rather than copying slightly different versions everywhere.

##### 22. Key Learnings

1. Raw images require consistent preprocessing before they can be
   used for neural-network training.

2. Images commonly need resizing so batches contain tensors with
   consistent dimensions.

3. Resizing must be chosen carefully because changing aspect ratio
   can distort visual information.

4. Image tensors commonly use:
   [channels, height, width].

5. A batch of images commonly uses:
   [batch_size, channels, height, width].

6. Scaling converts pixel values such as 0–255 into a smaller
   numerical range such as 0.0–1.0.

7. Normalization further transforms image values using expected
   channel statistics.

8. Pretrained models may require specific preprocessing and
   normalization consistent with their pretrained weights.

9. Data augmentation creates realistic variations of training images
   to improve model generalization.

10. Random augmentation should normally be applied to training data,
    while validation and test preprocessing should remain deterministic.

11. Augmentation is commonly performed dynamically during loading
    rather than permanently creating many new image files.

12. PyTorch Dataset objects load individual image/label pairs, while
    DataLoader objects create batches for training.

13. 🔍 XAI can help reveal whether a model learned meaningful defect
    features or inappropriate shortcuts such as backgrounds.

14. ⚖️ RAI requires attention to class representation, data leakage,
    image quality, realistic augmentation and production
    representativeness.

15. ⚖️ RAI reliability also requires consistent preprocessing between
    training and production inference.